# Practice 4-4. 희소 검색 (Sparse Retrieval) — PostgreSQL/pgvector로 직접 구현하는 BM25

책은 `langchain_community.retrievers.BM25Retriever`(Kiwi 형태소 분석 전처리)로 프로세스 메모리 안에 BM25 인덱스를 만듭니다.
이 방식은 코퍼스 전체를 메모리에 올려두고 매 질의마다 전수 스캔하므로, 억 단위 레코드에는 애초에 적용할 수 없습니다.

이 노트북은 같은 개념을 인프라 레벨에서 다시 만듭니다. PostgreSQL(pgvector 이미지 + `mecab-ko`/`textsearch_ko` 확장, Docker 컨테이너 `pgvector`)에
저장된 문서에 대해

1. 먼저 PostgreSQL이 기본 제공하는 전문 검색 랭킹 함수 `ts_rank_cd`로 스파스 검색의 최소 형태를 확인하고,
2. pgvector가 기본 지원하지 않는 **BM25를 직접 구현**합니다. `tsvector`에서 순색인(문서별 토큰 등장 횟수, TF)과
   역색인(토큰별 등장 문서 수, DF)을 추출해 별도 테이블로 관리하고, 트리거로 문서 추가·수정·삭제 시 이 색인들을 자동으로
   증분(increment/decrement) 갱신합니다. 쿼리 시점에는 저장된 통계만으로 BM25 점수를 계산하므로, 코퍼스가 커져도 매 질의마다
   원문을 다시 스캔할 필요가 없습니다.

이렇게 구성한 BM25 검색기는 `BaseRetriever`를 상속해 LangChain 파이프라인(`RetrievalQA`)에 그대로 연결합니다.

## 0. 환경 설정

In [ ]:
!pip install "psycopg[binary]"

In [ ]:
import os
import re
import psycopg

# --- LM Studio (답변 생성용 로컬 LLM) ---
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://...:.../v1")
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"   # LM Studio에 로드되어 있는 로컬 채팅 모델

# --- PostgreSQL (pgvector 이미지, mecab-ko/textsearch_ko 확장 포함, Docker 컨테이너) ---
PG_HOST = os.getenv("PG_HOST", "pgvector")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB = os.getenv("PG_DB", "admin")
PG_USER = os.getenv("PG_USER", "admin")
PG_PASSWORD = os.getenv("PG_PASSWORD", "...")
PG_DSN = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "투자설명서.pdf"


def strip_think(text: str) -> str:
    """qwen3 등 추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    # qwen3 계열 추론 모델은 답변 전에 <think> 추론 토큰을 많이 소모하므로 넉넉히 잡는다
    # (1024로는 추론만 하다가 잘려 답변이 빈 문자열로 나오는 경우가 있었다)
    max_tokens=2048,
)

# --- PostgreSQL 연결 확인 ---
with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print("PostgreSQL:", cur.fetchone()[0].split(",")[0])

# --- LLM 연결 확인 ---
print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])


## 1. 문서 로드 · 분할

책과 동일하게 `투자설명서.pdf`를 `PyPDFLoader`로 읽어 `RecursiveCharacterTextSplitter`(chunk_size=2000, chunk_overlap=200)로 분할합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(DATA_PATH)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = loader.load_and_split(doc_splitter)

print("분할된 문서 수:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:150])


## 2. PostgreSQL 스키마 구축

BM25를 매 질의마다 다시 계산하지 않고 **저장된 통계값을 조합**해서 구하려면 다음 세 가지가 필요합니다.

| 필요한 값 | 저장 위치 |
|---|---|
| 문서 길이(토큰 수), 코퍼스 평균 문서 길이 | 문서 테이블의 `doc_len` 컬럼 + 코퍼스 통계 테이블 |
| TF (문서 안에서 토큰이 몇 번 등장했는가) | 순색인(forward index) 테이블 — (문서, 토큰) 별 등장 횟수 |
| DF (토큰이 몇 개 문서에 등장했는가) → IDF | 역색인(inverted index) 테이블 — 토큰별 등장 문서 수 |

문서가 추가·삭제될 때마다 이 세 가지를 트리거로 자동 갱신하면, 색인 유지 비용은 "바뀐 문서 1건" 규모로 끝나고
코퍼스 전체를 다시 훑을 필요가 없습니다.

### 2.1 문서 테이블 — `tsvector` + GIN 인덱스

In [ ]:
DDL_DOCS_TABLE = """
CREATE EXTENSION IF NOT EXISTS textsearch_ko;

CREATE TABLE IF NOT EXISTS practice4_bm25_docs (
    id       SERIAL PRIMARY KEY,
    content  TEXT NOT NULL,
    page     INTEGER,
    tsv      TSVECTOR,      -- 형태소 분석 결과 (mecab-ko 기반 korean 설정)
    doc_len  INTEGER         -- 이 문서의 총 토큰 수 (BM25 문서 길이 정규화용)
);

CREATE INDEX IF NOT EXISTS idx_practice4_bm25_docs_tsv
    ON practice4_bm25_docs USING GIN (tsv);
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_DOCS_TABLE)
print("문서 테이블 준비 완료")


### 2.2 순색인(TF) · 역색인(DF) · 코퍼스 통계 테이블

In [ ]:
DDL_INDEX_TABLES = """
-- 순색인(forward index): (문서, 토큰) 조합별 등장 횟수(TF)
CREATE TABLE IF NOT EXISTS practice4_bm25_term_freq (
    doc_id  INTEGER NOT NULL REFERENCES practice4_bm25_docs(id) ON DELETE CASCADE,
    token   TEXT NOT NULL,
    tf      INTEGER NOT NULL,
    PRIMARY KEY (doc_id, token)
);
CREATE INDEX IF NOT EXISTS idx_practice4_bm25_term_freq_token
    ON practice4_bm25_term_freq (token);

-- 역색인(inverted index): 토큰별 등장 문서 수(DF)
CREATE TABLE IF NOT EXISTS practice4_bm25_term_df (
    token  TEXT PRIMARY KEY,
    df     INTEGER NOT NULL
);

-- 코퍼스 통계: 문서 수 + 총 토큰 수(단일 행) → 평균 문서 길이 = total_len / doc_count
CREATE TABLE IF NOT EXISTS practice4_bm25_stats (
    id         SMALLINT PRIMARY KEY DEFAULT 1,
    doc_count  INTEGER NOT NULL DEFAULT 0,
    total_len  BIGINT  NOT NULL DEFAULT 0,
    CHECK (id = 1)
);
INSERT INTO practice4_bm25_stats (id, doc_count, total_len)
VALUES (1, 0, 0)
ON CONFLICT (id) DO NOTHING;
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_INDEX_TABLES)
print("순색인/역색인/통계 테이블 준비 완료")


### 2.3 트리거 — 문서 추가·수정·삭제 시 색인을 자동으로 증분 갱신

- **BEFORE INSERT/UPDATE**: `content` → `tsv`(형태소 분석), `doc_len`(총 토큰 수) 계산
- **AFTER INSERT**: 새 문서의 토큰들을 순색인에 추가하고, 역색인의 DF를 +1, 코퍼스 통계를 갱신
- **AFTER DELETE**: 순색인에서 해당 문서 행을 지우고, 역색인의 DF를 -1(0이 되면 토큰 자체를 제거), 코퍼스 통계를 갱신
- **AFTER UPDATE**: DELETE 효과(OLD 제거) + INSERT 효과(NEW 반영)를 순서대로 적용

DF 증분·감소는 **UPDATE로 값을 내린 뒤 별도의 DELETE 문으로 0 이하인 토큰을 정리**하는 두 단계로 나눴습니다.
(하나의 `WITH ... UPDATE ... RETURNING` 뒤에 같은 테이블을 대상으로 `DELETE`를 잇는 방식은, PostgreSQL이 같은 문장 안의
데이터 변경 CTE들을 같은 스냅숏에서 평가하기 때문에 방금 UPDATE한 행을 DELETE가 찾지 못하는 문제가 있어 사용하지 않았습니다.)

In [ ]:
DDL_TRIGGERS = """
CREATE OR REPLACE FUNCTION practice4_bm25_docs_before() RETURNS TRIGGER AS $$
BEGIN
    NEW.tsv := to_tsvector('korean', NEW.content);
    SELECT COALESCE(SUM(cardinality(positions)), 0)
      INTO NEW.doc_len
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights);
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_before
    BEFORE INSERT OR UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_before();


CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_insert() RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count + 1,
           total_len = total_len + NEW.doc_len
     WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_insert
    AFTER INSERT ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_insert();


CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_delete() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats
       SET doc_count = doc_count - 1,
           total_len = total_len - OLD.doc_len
     WHERE id = 1;
    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_delete
    AFTER DELETE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_delete();


CREATE OR REPLACE FUNCTION practice4_bm25_docs_after_update() RETURNS TRIGGER AS $$
BEGIN
    DELETE FROM practice4_bm25_term_freq WHERE doc_id = OLD.id;

    UPDATE practice4_bm25_term_df
       SET df = df - 1
     WHERE token IN (SELECT lexeme FROM unnest(OLD.tsv) AS u(lexeme, positions, weights));

    DELETE FROM practice4_bm25_term_df WHERE df <= 0;

    UPDATE practice4_bm25_stats SET doc_count = doc_count - 1, total_len = total_len - OLD.doc_len WHERE id = 1;

    INSERT INTO practice4_bm25_term_freq (doc_id, token, tf)
    SELECT NEW.id, lexeme, cardinality(positions)
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (doc_id, token) DO UPDATE SET tf = EXCLUDED.tf;

    INSERT INTO practice4_bm25_term_df (token, df)
    SELECT lexeme, 1
      FROM unnest(NEW.tsv) AS u(lexeme, positions, weights)
    ON CONFLICT (token) DO UPDATE SET df = practice4_bm25_term_df.df + 1;

    UPDATE practice4_bm25_stats SET doc_count = doc_count + 1, total_len = total_len + NEW.doc_len WHERE id = 1;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE OR REPLACE TRIGGER trg_practice4_bm25_docs_after_update
    AFTER UPDATE ON practice4_bm25_docs
    FOR EACH ROW EXECUTE FUNCTION practice4_bm25_docs_after_update();
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_TRIGGERS)
print("트리거 준비 완료")


## 3. 문서 적재 (재실행 안전)

`DELETE`로 기존 문서를 지우면 `AFTER DELETE` 트리거가 순색인·역색인·통계를 원래대로 되돌립니다(`TRUNCATE`는 행 단위 트리거가 실행되지
않으므로 일부러 쓰지 않았습니다). 그 다음 새로 분할한 문서를 다시 넣으면 몇 번을 다시 실행해도 항상 같은 결과가 됩니다.

In [ ]:
with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("DELETE FROM practice4_bm25_docs;")
        cur.execute("ALTER SEQUENCE practice4_bm25_docs_id_seq RESTART WITH 1;")
        for d in docs:
            cur.execute(
                "INSERT INTO practice4_bm25_docs (content, page) VALUES (%s, %s)",
                (d.page_content, d.metadata.get("page")),
            )
    conn.commit()

with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT doc_count, total_len, total_len::float / GREATEST(doc_count,1) FROM practice4_bm25_stats;")
        doc_count, total_len, avg_len = cur.fetchone()
        cur.execute("SELECT count(*) FROM practice4_bm25_term_freq;")
        term_freq_rows = cur.fetchone()[0]
        cur.execute("SELECT count(*) FROM practice4_bm25_term_df;")
        vocab_size = cur.fetchone()[0]

print(f"적재된 문서 수 (doc_count) : {doc_count}")
print(f"총 토큰 수 (total_len)     : {total_len}")
print(f"평균 문서 길이 (avgdl)     : {avg_len:.2f}")
print(f"순색인 행 수 (문서x토큰)   : {term_freq_rows}")
print(f"역색인 어휘 수 (고유 토큰) : {vocab_size}")

# --- 재실행 안전성 점검: DELETE가 트리거를 통해 모든 파생 테이블을 정확히 되돌렸는지 단언 ---
assert doc_count == len(docs), f"적재된 문서 수({doc_count})가 분할 문서 수({len(docs)})와 다릅니다"
assert term_freq_rows > 0 and vocab_size > 0, "순색인/역색인이 비어 있습니다"
print("\n재실행 안전성 점검 통과: 이 셀 이전의 DELETE가 트리거로 모든 파생 테이블을 원래대로 되돌렸으므로,")
print("이 노트북을 몇 번을 다시 실행해도 위 통계는 항상 이 값으로 재구축됩니다.")


### 3.1 재실행 안전성 직접 증명 — 적재를 한 번 더 반복

말로만 "재실행해도 안전하다"고 하지 않고, 이 노트북 안에서 **같은 적재 절차를 한 번 더** 실행해 통계가 바뀌지 않는지 직접
확인합니다. 이 셀은 위 3장의 DELETE→INSERT 절차를 그대로 반복합니다: 기존 522건을 지우면(트리거가 파생 테이블을 전부
원상복구) 순색인·역색인·통계가 0으로 돌아가고, 다시 넣으면 정확히 같은 통계로 재구축됩니다.

In [ ]:
def reload_documents_and_get_stats():
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("DELETE FROM practice4_bm25_docs;")
            cur.execute("ALTER SEQUENCE practice4_bm25_docs_id_seq RESTART WITH 1;")
            for d in docs:
                cur.execute(
                    "INSERT INTO practice4_bm25_docs (content, page) VALUES (%s, %s)",
                    (d.page_content, d.metadata.get("page")),
                )
        conn.commit()

    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT doc_count, total_len FROM practice4_bm25_stats;")
            dc, tl = cur.fetchone()
            cur.execute("SELECT count(*) FROM practice4_bm25_term_freq;")
            tf_rows = cur.fetchone()[0]
            cur.execute("SELECT count(*) FROM practice4_bm25_term_df;")
            vocab = cur.fetchone()[0]
    return dc, tl, tf_rows, vocab


before = (doc_count, total_len, term_freq_rows, vocab_size)
after = reload_documents_and_get_stats()

print("1차 적재 통계 (doc_count, total_len, term_freq_rows, vocab_size):", before)
print("2차 적재 통계 (doc_count, total_len, term_freq_rows, vocab_size):", after)

assert before == after, "재적재 후 통계가 달라졌습니다 — 트리거의 증분 갱신 로직을 확인하세요"
print("\n두 번째 적재 결과가 첫 번째와 정확히 일치합니다 — DELETE→INSERT를 몇 번 반복해도 안전합니다.")


## 4. 희소 검색 기준선 — `ts_rank_cd`

pgvector(PostgreSQL)는 BM25를 기본 제공하지 않지만, `tsvector`/`tsquery`와 GIN 인덱스만 있으면 바로 쓸 수 있는
전문 검색 랭킹 함수 `ts_rank_cd`(cover density ranking)는 기본 제공합니다. 문서 길이 정규화나 IDF 포화(saturation) 같은
BM25 고유의 특성은 없지만, 별도 색인 구축 없이 가장 빠르게 쓸 수 있는 스파스 검색 기준선입니다.

In [ ]:
DDL_TSRANK_FUNC = """
CREATE OR REPLACE FUNCTION practice4_tsrank_search(q text, k integer DEFAULT 5)
RETURNS TABLE(doc_id integer, content text, page integer, score float4) AS $$
    SELECT id, content, page, ts_rank_cd(tsv, plainto_tsquery('korean', q)) AS score
      FROM practice4_bm25_docs
     WHERE tsv @@ plainto_tsquery('korean', q)
     ORDER BY score DESC
     LIMIT k;
$$ LANGUAGE sql STABLE;
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_TSRANK_FUNC)
print("practice4_tsrank_search() 함수 준비 완료")


In [ ]:
QUESTION = "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"

with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT doc_id, page, score, left(content, 60) FROM practice4_tsrank_search(%s, %s)", (QUESTION, 5))
        rows = cur.fetchall()

print(f"ts_rank_cd 결과 {len(rows)}건")
for doc_id, page, score, snippet in rows:
    print(f"  [doc_id={doc_id} page={page} score={score:.4f}] {snippet}")


`plainto_tsquery`는 질의에 담긴 토큰을 **모두 포함(AND)**하는 문서만 찾습니다. 그래서 질의 토큰 중 일부만 포함한 문서는
아예 후보에서 빠지고, 위 질의처럼 결과가 0건이 될 수도 있습니다("회사", "발행", "주식", "총", "발행량"을 전부 담은 문단이
코퍼스에 없기 때문입니다). BM25는 이와 달리 토큰이 일부만 일치해도 점수를 누적해서 매기므로(OR에 가까운 동작) 더 관대하고,
TF 포화·문서 길이 정규화까지 반영합니다.

## 5. 직접 구현하는 BM25

Okapi BM25 공식은 다음과 같습니다.

$$
\text{score}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}
$$

$$
\text{IDF}(t) = \ln\!\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)
$$

- $f(t, d)$ : TF, 문서 $d$ 안에서 토큰 $t$의 등장 횟수 → **순색인 테이블**(`practice4_bm25_term_freq`)에서 조회
- $n(t)$ : DF, 토큰 $t$가 등장하는 문서 수 → **역색인 테이블**(`practice4_bm25_term_df`)에서 조회
- $N$, $\text{avgdl}$ : 전체 문서 수, 평균 문서 길이 → **코퍼스 통계 테이블**(`practice4_bm25_stats`)에서 조회
- $k_1$(기본 1.2), $b$(기본 0.75) : TF 포화·문서 길이 정규화 강도를 조절하는 하이퍼파라미터

앞서 트리거로 이미 채워둔 세 테이블만 조합하면 되므로, 질의 시점에 원문을 다시 스캔하지 않습니다.

In [ ]:
DDL_BM25_FUNC = """
CREATE OR REPLACE FUNCTION practice4_bm25_search(
    q       text,
    k       integer DEFAULT 5,
    bm25_k1 float8  DEFAULT 1.2,
    bm25_b  float8  DEFAULT 0.75
) RETURNS TABLE(doc_id integer, content text, page integer, score float8) AS $$
    WITH stats AS (
        SELECT doc_count::float8 AS n,
               CASE WHEN doc_count > 0 THEN total_len::float8 / doc_count ELSE 0 END AS avgdl
          FROM practice4_bm25_stats WHERE id = 1
    ),
    qtokens AS (
        SELECT DISTINCT lexeme AS token
          FROM unnest(to_tsvector('korean', q)) AS u(lexeme, positions, weights)
    ),
    qidf AS (
        SELECT qt.token,
               ln( (s.n - COALESCE(d.df, 0) + 0.5) / (COALESCE(d.df, 0) + 0.5) + 1 ) AS idf
          FROM qtokens qt
          CROSS JOIN stats s
          LEFT JOIN practice4_bm25_term_df d ON d.token = qt.token
    ),
    scored AS (
        SELECT tf.doc_id,
               SUM( qidf.idf * (tf.tf * (bm25_k1 + 1))
                    / (tf.tf + bm25_k1 * (1 - bm25_b + bm25_b * doc.doc_len / NULLIF(s.avgdl, 0)))
                  ) AS score
          FROM practice4_bm25_term_freq tf
          JOIN qidf ON qidf.token = tf.token
          JOIN practice4_bm25_docs doc ON doc.id = tf.doc_id
          CROSS JOIN stats s
         GROUP BY tf.doc_id
    )
    SELECT scored.doc_id, doc.content, doc.page, scored.score
      FROM scored
      JOIN practice4_bm25_docs doc ON doc.id = scored.doc_id
     ORDER BY scored.score DESC
     LIMIT k;
$$ LANGUAGE sql STABLE;
"""

with psycopg.connect(PG_DSN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_BM25_FUNC)
print("practice4_bm25_search() 함수 준비 완료")


In [ ]:
with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT doc_id, page, score, left(content, 60) FROM practice4_bm25_search(%s, %s)", (QUESTION, 5))
        rows = cur.fetchall()

print(f"BM25 결과 {len(rows)}건")
for doc_id, page, score, snippet in rows:
    print(f"  [doc_id={doc_id} page={page} score={score:.4f}] {snippet}")


`ts_rank_cd`와 달리 BM25는 질의 토큰 중 일부만 겹치는 문서도 점수를 매겨 후보에 포함시키고, 문서 길이·토큰 포화까지 반영한 순위를 매깁니다.

## 6. LangChain 리트리버로 래핑

`BaseRetriever`를 상속해 `practice4_bm25_search()` 함수를 호출하는 리트리버를 만듭니다. 랭킹 연산은 전부 DB 서버에서
수행되고 파이썬 프로세스는 상위 k건만 받으므로, 책의 `BM25Retriever`처럼 코퍼스를 메모리에 올려두지 않습니다.

In [ ]:
from typing import List
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun


class PgSqlFunctionRetriever(BaseRetriever):
    """PostgreSQL 함수(ts_rank_cd 또는 직접 구현한 BM25)를 호출해 검색하는 리트리버.
    sql_func는 이 노트북에서 정의한 함수명 중 하나로만 내부에서 지정되며 쿼리 문자열은 항상 파라미터 바인딩으로 전달한다."""

    dsn: str
    sql_func: str
    k: int = 4

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        with psycopg.connect(self.dsn) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    f"SELECT doc_id, content, page, score FROM {self.sql_func}(%s, %s)",
                    (query, self.k),
                )
                rows = cur.fetchall()
        return [
            Document(
                page_content=content,
                metadata={"source": DATA_PATH, "doc_id": doc_id, "page": page, "score": float(score)},
            )
            for doc_id, content, page, score in rows
        ]


# 책의 bm25_retriever.k = 2 에 대응
bm25_retriever = PgSqlFunctionRetriever(dsn=PG_DSN, sql_func="practice4_bm25_search", k=2)

found = bm25_retriever.invoke(QUESTION)
print(f"검색 결과 {len(found)}건")
for d in found:
    print(" -", d.metadata, d.page_content[:50])


## 7. 답변 생성 (RetrievalQA)

책과 동일하게 검색된 문서를 `RetrievalQA`(`chain_type="stuff"`)에 넣어 최종 답변을 생성합니다.

In [ ]:
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=bm25_retriever,
    return_source_documents=True,
)

result = qa_chain.invoke({"query": QUESTION})

print("질문:", QUESTION)
print("답변:", strip_think(result["result"]))
print("\n사용된 문서:")
for d in result["source_documents"]:
    print(" -", d.metadata)


## 8. 관찰 — 키워드 검색만으로는 한계가 있다

실제로 이 코퍼스에는 "가중평균유통보통주식수는 13,602,977주 입니다"(435쪽)와 "발행주식총수(A) 보통주 13,602,977"(489쪽)처럼
정답이 담긴 문단이 존재합니다. 하지만 질의 "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"는 이 문단들과 겹치는 단어가
많지 않고("발행량"이라는 표현 자체가 원문에는 그대로 등장하지 않습니다), BM25 상위 5건 중에도 해당 문단(489쪽, 4위)이
`k=2`로는 잘려나갑니다. 즉 **질의와 문서의 표현이 다르면 순수 키워드 검색은 정답 문단을 놓칠 수 있습니다.**

다음 노트북(`05-pgvector-hnsw-dense-retrieval.ipynb`)에서는 의미 기반 임베딩으로 이런 표현 차이를 넘어서는 밀집 검색을, 그다음(`06-bm25-pgvector-ensemble-retrieval.ipynb`)에서는
이 노트북의 BM25와 밀집 검색을 함께 쓰는 앙상블 검색을 pgvector로 구현합니다.